# Benchmark PyMC Initialization & Sampler Backends

Isolates where time actually goes before useful posterior draws start:

1. **Cold JAX/XLA compile** of the diffrax ODE solve (`SolOp`/`VJPSolOp`) vs. warm re-evaluation — this happens once per *process*, so it is the dominant cost whenever a sampler spawns multiple processes (multiprocessing chains).
2. **`pm.init_nuts`** across initialization strategies, on an already-warmed model, to isolate pure NUTS warmup/mass-matrix-adaptation cost from compile cost. Note: `pm.init_nuts` is only used by `nuts_sampler="pymc"` — `nutpie` and `numpyro` run their own internal warmup and never call it.
3. **Sampler engines** (`pymc`, `nutpie` w/ numba backend, `nutpie` w/ jax backend, `numpyro`) for total time-to-first-draw on a single chain/core, so multiprocessing recompilation doesn't confound the comparison.

Reuses the private model-building helpers from `inference_runner.py` directly so the model is built once and reused across every timing run.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "../Utilities")

import signal
import time
from contextlib import contextmanager
from pathlib import Path

import pymc as pm
import blackjax

from inference_runner import (
    import_solver_params,
    _build_simulator,
    _build_pytensor_sol_op,
    _build_pymc_model,
)
from experiment_framework import load_experiment_bundle, validate_experiment_config
from reaction_model_builder import build_ode_system_from_reactions

BENCHMARK_TIMEOUT_S = 120  # skip any method that takes longer than 2 minutes

class _TimedOut(Exception):
    pass

@contextmanager
def time_limit(seconds):
    def _handler(signum, frame):
        raise _TimedOut(f"skipped — exceeded {seconds}s limit")
    prev = signal.signal(signal.SIGALRM, _handler)
    signal.alarm(seconds)
    try:
        yield
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, prev)

Using 8 CPU core(s) for JAX/PyMC
Total JAX local devices initialized: 8


In [7]:
folder_name = "Full FAS FabD kon"
tune_steps = 10

solver_param_path = Path(f"../Results/{folder_name}/solver_params.json")
imported = import_solver_params(solver_param_path)
solver_params = imported.solver_params

ode_system, species_names, param_names, param_values, scaling_params = build_ode_system_from_reactions(
    imported.reactions_source
)
validate_experiment_config(
    solver_params=solver_params,
    solver_params_file=str(solver_param_path),
    species_names=species_names,
)
experiment = load_experiment_bundle(
    solver_params=solver_params,
    solver_params_file=str(solver_param_path),
    species_names=species_names,
)

simulator = _build_simulator(
    ode_system=ode_system,
    species_names=species_names,
    solver_params=solver_params,
    experiment=experiment,
)
sol_op = _build_pytensor_sol_op(simulator)

pm_model, free_params, default_initvals = _build_pymc_model(
    solver_params=solver_params,
    sol_op=sol_op,
    param_names=param_names,
    param_values=param_values,
    scaling_params=scaling_params,
    observed_data=experiment.observed_values,
    observed_sigma=experiment.observed_sigma,
)

print(f"Free parameters: {free_params}")
print(f"Species count: {len(species_names)}")

Free parameters: ['kon_D_MalCoA', 'kon_D_Act_ACP']
Species count: 320


## 1. Cold compile vs. warm evaluation

Compiles the model logp once, then calls it twice: the first call triggers JAX tracing/XLA compilation of the diffrax solve inside `SolOp.perform`; the second call reuses the cached compiled function. The gap between these two numbers is the per-process tax any multiprocessing sampler pays on every chain.

In [8]:
with pm_model:
    logp_fn = pm_model.compile_logp()

initial_point = pm_model.initial_point()

t0 = time.perf_counter()
logp_fn(initial_point)
t_cold = time.perf_counter() - t0

t0 = time.perf_counter()
logp_fn(initial_point)
t_warm = time.perf_counter() - t0

print(f"Cold (compile + first eval): {t_cold:8.2f} s")
print(f"Warm (cached, repeat eval):  {t_warm:8.4f} s")
print(f"Compile overhead:            {t_cold - t_warm:8.2f} s")

Cold (compile + first eval):    64.66 s
Warm (cached, repeat eval):   43.9734 s
Compile overhead:               20.69 s


## 2. `pm.init_nuts` across initialization strategies

Run on the model that was already warmed above, so every strategy pays the same (near-zero) compile cost and the timing differences reflect only the warmup/mass-matrix-adaptation work `pm.init_nuts` itself does. This only applies to the `nuts_sampler="pymc"` path — `nutpie`/`numpyro` skip this function entirely.

In [4]:
init_strategies = ["jitter+adapt_diag", "adapt_diag", "jitter+adapt_full", "advi+adapt_diag", "advi"]

init_benchmark_results = []
for strategy in init_strategies:
    t0 = time.perf_counter()
    try:
        with time_limit(BENCHMARK_TIMEOUT_S):
            pm.init_nuts(
                init=strategy,
                chains=1,
                tune=tune_steps,
                model=pm_model,
            )
        elapsed = time.perf_counter() - t0
        init_benchmark_results.append((strategy, elapsed, None))
    except (_TimedOut, Exception) as exc:
        elapsed = time.perf_counter() - t0
        init_benchmark_results.append((strategy, elapsed, str(exc)))

print(f"{'strategy':>20s}  {'time (s)':>10s}  status")
for strategy, elapsed, error in init_benchmark_results:
    status = "OK" if error is None else f"FAILED: {error}"
    print(f"{strategy:>20s}  {elapsed:10.2f}  {status}")

Initializing NUTS using jitter+adapt_diag...
Initializing NUTS using adapt_diag...
Initializing NUTS using jitter+adapt_full...
/Users/annettethompson/anaconda3/envs/Bayesian/lib/python3.13/site-packages/pymc/step_methods/hmc/quadpotential.py:760: UserWarning: QuadPotentialFullAdapt is an experimental feature
  warnings.warn("QuadPotentialFullAdapt is an experimental feature")
Initializing NUTS using advi+adapt_diag...


Output()

Initializing NUTS using advi...


Output()

            strategy    time (s)  status
   jitter+adapt_diag       29.67  OK
          adapt_diag        0.88  OK
   jitter+adapt_full        1.03  OK
     advi+adapt_diag      120.05  FAILED: skipped — exceeded 120s limit
Apply node that caused the error: VJPSolOp(Exp.0, 11397.897904000001, 1.0, 60222.860002345195, 402.13929190000005, 1.0, 1.0, Exp.0, 3091.6798064600002, 1411.1030972777992, 402.13929190000005, 1.0, Composite{...}.1)
Toposort index: 37
Inputs types: [TensorType(float64, shape=()), TensorType(float64, shape=()), TensorType(float64, shape=()), TensorType(float64, shape=()), TensorType(float64, shape=()), TensorType(float64, shape=()), TensorType(float64, shape=()), TensorType(float64, shape=()), TensorType(float64, shape=()), TensorType(float64, shape=()), TensorType(float64, shape=()), TensorType(float64, shape=()), TensorType(float64, shape=(None, 22))]
Inputs shapes: [(), (), (), (), (), (), (), (), (), (), (), (), (1, 22)]
Inputs strides: [(), (), (), (), (), (), ()

## 3. Sampler engine comparison (single chain, single core)

Times `pm.sample` with `draws=1`, a short `tune`, and `chains=cores=1` for each engine, so every number is dominated by that engine's own compile + warmup path rather than multiprocessing overhead. `nutpie` defaults to its numba backend, which has no numba dispatch registered for `SolOp`/`VJPSolOp` and silently falls back to slow Python object-mode for the ODE node (that's what the warning filter at the top of `inference_runner.py` is masking).

In [ ]:
sampler_configs = [
    ("pymc", dict(nuts_sampler="pymc", init="adapt_diag")),
    ("nutpie", dict(nuts_sampler="nutpie")),
    ("numpyro", dict(nuts_sampler="numpyro")),
]

sampler_benchmark_results = []
for label, kwargs in sampler_configs:
    t0 = time.perf_counter()
    try:
        with time_limit(BENCHMARK_TIMEOUT_S):
            with pm_model:
                pm.sample(
                    draws=1,
                    tune=tune_steps,
                    chains=1,
                    cores=1,
                    random_seed=0,
                    progressbar=False,
                    discard_tuned_samples=True,
                    compute_convergence_checks=False,
                    **kwargs,
                )
        elapsed = time.perf_counter() - t0
        sampler_benchmark_results.append((label, elapsed, None))
    except (_TimedOut, Exception) as exc:
        elapsed = time.perf_counter() - t0
        sampler_benchmark_results.append((label, elapsed, str(exc)))

print(f"{'engine':>35s}  {'time (s)':>10s}  status")
for label, elapsed, error in sampler_benchmark_results:
    status = "OK" if error is None else f"FAILED: {error}"
    print(f"{label:>35s}  {elapsed:10.2f}  {status}")


Only 1 samples per chain. Reliable r-hat and ESS diagnostics require longer chains for accurate estimate.
Initializing NUTS using adapt_diag...
Sequential sampling (1 chains in 1 job)
NUTS: [kon_D_MalCoA, kon_D_Act_ACP]
Sampling 1 chain for 10 tune and 1 draw iterations (10 + 1 draws total) took 23 seconds.
Only 1 samples per chain. Reliable r-hat and ESS diagnostics require longer chains for accurate estimate.
Only 1 samples per chain. Reliable r-hat and ESS diagnostics require longer chains for accurate estimate.


                             engine    time (s)  status
                pymc (default NUTS)       34.47  OK
    nutpie (numba backend, default)       20.04  OK
                            numpyro       50.45  OK


## 4. Metropolis vs. HMC

Metropolis methods don't compute gradients — they only call the forward ODE solve (no VJP), so each *proposal* is cheaper than an HMC step. The tradeoff is that they accept far fewer proposals as dimensionality grows and mix much more slowly, often requiring orders-of-magnitude more draws to reach the same ESS.

With only 2 free parameters this model is a rare case where gradient-free methods can be competitive in wall-clock time, so it is worth measuring directly:

- **`Metropolis`** — random-walk proposals, one forward solve per proposal, accepts/rejects on log-likelihood ratio.
- **`DEMetropolis`** — Differential Evolution variant; runs multiple chains and uses inter-chain differences to propose moves. More efficient per-chain than plain Metropolis for correlated posteriors but requires `chains ≥ 2`.

In [11]:
cores = 3
draws = 3
tunes = 10
mh_benchmark_results = []

# Metropolis: drive step.step(point) directly — no pm.sample overhead, no multiprocessing.
# step must be created inside the model context.
t0 = time.perf_counter()

with pm_model:
    ip = pm_model.initial_point()
    step = pm.Metropolis(initial_point=ip)
    point = ip
    for _ in range(tunes):
        point, _ = step.step(point)
elapsed = time.perf_counter() - t0
mh_benchmark_results.append(("Metropolis", tunes, elapsed, None))

# DEMetropolis: needs pm.sample to initialize and manage the chain population.
# Requires chains >= 3 (picks 2 other chains for each proposal).
# Step must be created inside the model context.
t0 = time.perf_counter()

with pm_model:
    pm.sample(
        draws=draws,
        tune=tunes,
        chains=cores,
        cores=cores,
        step=pm.DEMetropolis(),
        random_seed=0,
        progressbar=False,
        discard_tuned_samples=True,
        compute_convergence_checks=False,
    )
elapsed = time.perf_counter() - t0
mh_benchmark_results.append((f"DEMetropolis ({cores} chains)", (tunes+draws) * cores, elapsed, None))

print(f"{'method':>25s}  {'steps':>6s}  {'total (s)':>10s}  {'s/step':>8s}  status")
for label, n_steps, elapsed, error in mh_benchmark_results:
    per_step = elapsed / n_steps if error is None else float("nan")
    status = "OK" if error is None else f"FAILED: {error}"
    print(f"{label:>25s}  {n_steps:>6d}  {elapsed:>10.2f}  {per_step:>8.4f}  {status}")

try:
    print(f"\nReference — warm forward+VJP eval (section 1): {t_warm:.4f} s")
    print("Metropolis step = 1 forward eval only (no VJP).")
    print("HMC/NUTS step   = ~L gradient evals (forward + adjoint), L = leapfrog depth.")
except NameError:
    print("\n(Run section 1 first to see the warm-eval reference time.)")

Only 3 samples per chain. Reliable r-hat and ESS diagnostics require longer chains for accurate estimate.
Population sampling (3 chains)
DEMetropolis: [kon_D_MalCoA, kon_D_Act_ACP]
Attempting to parallelize chains to all cores. You can turn this off with `pm.sample(cores=1)`.
Population parallelization failed. Falling back to sequential stepping of chains.
Sampling 3 chains for 10 tune and 3 draw iterations (30 + 9 draws total) took 179265 seconds.


                   method   steps   total (s)    s/step  status
               Metropolis      10     2445.79  244.5792  OK
  DEMetropolis (3 chains)      39     8291.42  212.6006  OK

Reference — warm forward+VJP eval (section 1): 43.9734 s
Metropolis step = 1 forward eval only (no VJP).
HMC/NUTS step   = ~L gradient evals (forward + adjoint), L = leapfrog depth.


## Reading the results

**Section 1 — compile overhead:**
The gap between cold and warm is the per-process tax any multiprocessing sampler pays on every chain. For `nutpie`/default `pymc` it's multiplied by `chains`; `numpyro` pays it once (single process, chains vmapped).

**Section 2 — `pm.init_nuts` strategies:**
Little spread across strategies is expected here — 2 free parameters means mass-matrix adaptation complexity is negligible. The dominant cost in every case is the ODE solve per gradient eval, not NUTS bookkeeping.

**Section 3 — HMC-family engine comparison:**
- `nutpie` silently falls back to Python object-mode for `SolOp`/`VJPSolOp` 
- `numpyro` is the strongest choice for multi-chain runs if section 1 shows large compile overhead, since it pays that cost once regardless of chain count.

**Section 4 — Metropolis vs. HMC:**
Metropolis proposals cost ~1 forward ODE solve each (no VJP). NUTS steps cost ~L gradient evals (forward + adjoint), where L is the leapfrog depth NUTS auto-tunes. For this model the question is whether the higher acceptance rate and faster mixing of NUTS gives more *effective* samples per wall-clock second than Metropolis's cheaper-but-more-correlated steps. The time/step column makes this concrete: if Metropolis's time/step is much less than `t_warm` from section 1 (which includes a VJP), Metropolis has a real per-step advantage — then the only question is how many more steps it needs to achieve the same ESS.